# Reproduce Paper Figures and Tables

This notebook generates all figures and tables from the paper:
**"Machine Learning for Aquatic Ecotoxicity Prediction with Quantified Uncertainty"**

### Prerequisites
1. Install dependencies: `pip install -r requirements.txt`
2. Place ADORE data in `data/raw/` (see `data/raw/README.md`)
3. Train the model: `python scripts/train_bfm.py`
4. Generate predictions: `python scripts/generate_predictions.py`

Once the trained model artifacts exist in `outputs/models/`, this notebook reproduces every figure and table.

## Setup

In [ ]:
import sys
from pathlib import Path

# Project root (one level up from notebooks/)
ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR / "src"))
sys.path.insert(0, str(ROOT_DIR / "analysis"))

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

## Configuration

Adjust these parameters to tweak the analysis. Changes here propagate to all figures below.

In [ ]:
# Duration to filter to for most analyses
DURATION_HOURS = 48

# Chemicals for SSD case studies (Figures 7-9)
SSD_CHEMICALS = {
    "Atrazine": "1912-24-9",
    "Chlorfenprop-methyl": "14437-17-3",
}

# HC percentile for Figures 10-11
HC_PERCENTILE = 20

# Uncertainty example plots (Figure 5)
N_EXAMPLE_CHEMICALS = 5
MIN_OBS_FOR_EXAMPLES = 10
RANDOM_SEED = 42

# MCMC settings for SSD uncertainty (Figure 7)
N_SSD_CURVES = 2000

# Figure output settings
SAVE_FIGURES = True  # Set to False to just display without saving
FIGURE_DPI = 150

## Load Data

Load the ADORE dataset and model outputs once; reused by all figures.

In [ ]:
import numpy as np
import pandas as pd
from data.load_ecotox import load_ecotox_data

DATA_DIR = ROOT_DIR / "data" / "raw"
MODELS_DIR = ROOT_DIR / "outputs" / "models"

# Load raw dataset
full_data, y_centered, y_mean = load_ecotox_data(
    adore_path=DATA_DIR / "ecotox_mortality_processed.csv",
    chemicals_path=DATA_DIR / "ecotox_properties_with-oecd-function.csv",
    use_molar=False,
    use_selfies=False, use_mol2vec=False, use_fingerprint=False,
    shuffle=True, random_state=42,
)
full_data["y_true"] = y_centered + y_mean

print(f"Loaded {len(full_data):,} observations")
print(f"  Unique chemicals: {full_data['CAS'].nunique():,}")
print(f"  Unique species:   {full_data['species'].nunique():,}")
print(f"  Centering mean:   {y_mean:.4f} log mg/L")

In [ ]:
# Load out-of-fold predictions (from cross-validation)
oof_mean = np.load(MODELS_DIR / "oof_mean.npy")
oof_epistemic = np.load(MODELS_DIR / "oof_epistemic.npy")
oof_aleatoric = np.load(MODELS_DIR / "oof_aleatoric.npy")

df = full_data.copy()
df["y_pred"] = oof_mean + y_mean
df["epistemic_var"] = oof_epistemic
df["aleatoric_var"] = oof_aleatoric
df["total_var"] = df["epistemic_var"] + df["aleatoric_var"]
df["epistemic_sd"] = np.sqrt(df["epistemic_var"])
df["aleatoric_sd"] = np.sqrt(df["aleatoric_var"])
df["total_sd"] = np.sqrt(df["total_var"])

print("Loaded OOF predictions.")

In [ ]:
# Load full prediction matrix (from full-dataset training)
pred_df = pd.read_parquet(MODELS_DIR / "full_predictions.parquet")
pred_df["epistemic_sd"] = np.sqrt(pred_df["pred_epistemic_var"])
pred_df["aleatoric_sd"] = np.sqrt(pred_df["pred_aleatoric_var"])

print(f"Loaded {len(pred_df):,} full predictions.")

---
## Dataset Characterization

### Table 1: Summary Statistics

In [ ]:
from dataset_figures import table1_summary

table1_summary(full_data)

### Figure 1: Rank-Frequency Plots

In [ ]:
from dataset_figures import figure1_rank_frequency

figure1_rank_frequency(full_data)

### Figure 2: Distribution of RSDs

In [ ]:
from dataset_figures import figure2_rsd_distribution

figure2_rsd_distribution(full_data)

---
## Model Performance

### Figure 3: Predicted vs Measured Correlation

In [ ]:
from analyze_results import plot_predicted_vs_measured_correlation

corr_stats = plot_predicted_vs_measured_correlation(df, duration_hours=DURATION_HOURS)

### Figure 4: Residual Bias Analysis

In [ ]:
from analyze_results import analyze_prediction_bias

bias_stats = analyze_prediction_bias(df, duration_hours=DURATION_HOURS)

---
## Uncertainty Calibration

### Table 2: Aleatoric Calibration by Replicate Count

In [ ]:
from dataset_figures import table2_aleatoric_calibration

table2_aleatoric_calibration(full_data)

### Figure 5: Example Predictions with Uncertainty

In [ ]:
from uncertainty_figures import plot_example_predictions

plot_example_predictions(df)

### Figure 6: Uncertainty vs Data Availability

In [ ]:
from uncertainty_figures import plot_uncertainty_vs_observations

chem_stats = plot_uncertainty_vs_observations(df)

---
## Species Sensitivity Distributions (SSDs)

### Figures 7-9: SSD Analysis per Chemical

For each chemical in `SSD_CHEMICALS`:
- **Figure 7**: Ensemble SSDs from posterior samples (epistemic uncertainty band)
- **Figure 8**: Novel SSD with per-species uncertainty bars
- **Figure 9**: Traditional vs novel SSD overlay

In [ ]:
import ssd_analysis
from ssd_analysis import (
    set_target, filter_observations, filter_predictions,
    plot_traditional_ssd, plot_novel_ssd_with_uncertainty,
    plot_traditional_vs_novel_ssd,
)
from ssd_mc_uncertainty import plot_ssd_with_uncertainty

for chem_name, cas in SSD_CHEMICALS.items():
    print("\n" + "=" * 70)
    print(f"  {chem_name} (CAS {cas})")
    print("=" * 70)

    set_target(cas, chem_name)

    # Filter observations and predictions to this chemical
    df_obs_filtered = filter_observations(full_data)
    pred_filtered = filter_predictions(pred_df)

    # Figure 7: MCMC posterior ensemble SSD
    plot_ssd_with_uncertainty(pred_filtered, n_curves=N_SSD_CURVES)

    # Figure 8: Novel SSD with uncertainty bars
    plot_novel_ssd_with_uncertainty(pred_filtered)

    # Figure 9: Traditional vs novel comparison
    plot_traditional_vs_novel_ssd(df_obs_filtered, pred_filtered)

---
## HC20 Comparison (Figures 10-11)

This section computes HC20 values for all chemicals using MCMC posterior samples, then compares them with traditional lognormal-fitted HC20 values.

**Note:** This cell is computationally expensive (iterates over all chemicals x all posterior samples). It saves results to CSV so subsequent runs of the plotting cells can skip recomputation.

In [ ]:
from ssd_mc_uncertainty import compute_hcx_all_chemicals, compute_traditional_hcx

hcx_csv_path = ROOT_DIR / "outputs" / "figures" / "ssd_analysis" / f"hcx_comparison_{DURATION_HOURS}h.csv"

if hcx_csv_path.exists():
    print(f"Loading cached HCx comparison from {hcx_csv_path}")
    df_hcx = pd.read_csv(hcx_csv_path)
else:
    print("Computing HCx for all chemicals (this may take a while)...")
    df_mc = compute_hcx_all_chemicals(percentiles=[HC_PERCENTILE])

    df_trad = compute_traditional_hcx(full_data, percentiles=[HC_PERCENTILE])

    df_hcx = df_mc.merge(df_trad, on="CAS", how="left")
    df_hcx.to_csv(hcx_csv_path, index=False)
    print(f"Saved to {hcx_csv_path}")

print(f"HCx data: {len(df_hcx)} chemicals")

### Figure 10: HC20 Forest Plot

In [ ]:
from hcx_plots import figure10_forest_plot

figure10_forest_plot(df_hcx)

### Figure 11: HC20 Correlation (Traditional vs BFM)

In [ ]:
from hcx_plots import figure11_correlation_scatter

figure11_correlation_scatter(df_hcx)

---
## Summary

All figures and tables have been generated. Outputs are saved to `outputs/figures/`.